# Kairos v5 -- Final Selection Refactor
### Regime-Aware Financial Trend Classification
**Weighted CE | feature-group selection | ensemble check | anchored walk-forward**

This version keeps the corrected pipeline and shifts the notebook toward what the reruns actually supported:

- weighted cross-entropy remains the default for neural models
- no calibration in the main experiment path
- feature-group comparison is treated as a real model-selection stage
- the strongest proposed feature group is re-materialized as the final proposed run
- an optional ensemble block compares `Proposed + LSTM + XGBoost` against the single models
- optional multi-seed robustness remains available for the shortlisted models


---
## Section 1 -- Inputs and Data Pipeline

This notebook is now in **reporting freeze** mode.
The model stack is frozen and no further architecture, hyper-parameter, feature-selection,
or Fold 1 repair logic is performed here.

Pipeline overview:
1. Download S&P 500 and VIX data
2. Engineer the stationary features
3. Build fold specifications from sample end-dates
4. Select lambda inside each fold using training data only
5. Build aligned sequences ending at the label anchor day
6. Run the frozen final comparison on the selected standalone feature group
7. Generate reporting artifacts: ensemble diagnostics, confusion matrices, and multi-seed robustness
---


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — GPU CHECK + PACKAGE INSTALL                            ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print('✅ GPU detected:', result.stdout.strip())
except Exception:
    print('⚠️  No GPU. Go to Runtime → Change runtime type → T4 GPU')
    print('   Training will work on CPU but will be much slower.')

print('\nInstalling packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'yfinance', 'ta', 'xgboost'],
    check=True
)
print('✅ All packages installed.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — ALL IMPORTS                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

import copy
import itertools
import json
import math
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
)
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Imports complete. Device: {DEVICE}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA DOWNLOAD + FEATURE ENGINEERING (C = 11)           ║
# ╚══════════════════════════════════════════════════════════════════╝

TARGET_START = '2013-01-01'
DOWNLOAD_START = '2012-01-01'   # 1 year earlier for EMA-200 warmup
END_DATE   = '2023-12-31'
TICKER     = '^GSPC'
VIX_TICKER = '^VIX'
LOOKBACK   = 60     # UPGRADE 1: expanded from 20 → 60 (one fiscal quarter)
HORIZON    = 5
VOL_WINDOW = 30
N_FOLDS    = 3
TEST_RATIO = 0.12

print(f'Downloading {TICKER} ({DOWNLOAD_START} to {END_DATE})...')
raw = yf.download(TICKER, start=DOWNLOAD_START, end=END_DATE,
                  auto_adjust=True, progress=False)
raw = raw.dropna()
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
print(f'  Raw S&P 500: {len(raw)} trading days (incl. warm-up)')

print(f'Downloading {VIX_TICKER}...')
vix_raw = yf.download(VIX_TICKER, start=DOWNLOAD_START, end=END_DATE,
                      auto_adjust=True, progress=False)
vix_raw = vix_raw.dropna()
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)

# Align VIX to S&P 500 index; forward-fill any missing dates
vix_close = vix_raw['Close'].reindex(raw.index).ffill()
print(f'  VIX aligned: {vix_close.notna().sum()} valid dates')

feat  = pd.DataFrame(index=raw.index)
close = raw['Close']
open_ = raw['Open']
high  = raw['High']
low   = raw['Low']
vol   = raw['Volume']

# ── ORIGINAL 7 FEATURES ─────────────────────────────────────────────
# 1-4: OHLC log-returns (stationary by construction)
feat['log_ret_close'] = np.log(close / close.shift(1))
feat['log_ret_open']  = np.log(open_  / open_.shift(1))
feat['log_ret_high']  = np.log(high   / high.shift(1))
feat['log_ret_low']   = np.log(low    / low.shift(1))

# 5: 20-day rolling realized volatility
feat['rolling_vol_20'] = (
    feat['log_ret_close'].rolling(window=20, min_periods=20).std()
)

# 6: RSI-14 normalized to [0,1]
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_l = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs    = avg_g / (avg_l + 1e-10)
    return (100 - 100 / (1 + rs)) / 100.0

feat['rsi_14'] = compute_rsi(close)

# 7: MACD signal normalized by price (stationary)
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
feat['macd_norm'] = (ema12 - ema26) / close

# ── NEW 4 FEATURES (UPGRADE 1) ──────────────────────────────────────
# 8: log(VIX) — macro fear signal; log-transform for stationarity
feat['log_vix'] = np.log(vix_close.clip(lower=1e-6))

# 9: Volume Shock — 20-day rolling z-score of volume
#    Formula: (V_t - Mean_20) / Std_20  → stationary, no lookahead
vol_mean = vol.rolling(window=20, min_periods=20).mean()
vol_std  = vol.rolling(window=20, min_periods=20).std()
feat['vol_shock_20'] = (vol - vol_mean) / (vol_std + 1e-8)

# 10: Short-Term 5-Day Momentum — log(Close_t / Close_{t-5})
feat['momentum_5d'] = np.log(close / close.shift(5))

# 11: Long-Term Trend — EMA(50) vs EMA(200) distance normalized by price
#     Positive → uptrend (golden cross), Negative → downtrend (death cross)
#     EMA-200 needs 200 days warmup (guaranteed by DOWNLOAD_START shift)
ema50  = close.ewm(span=50,  adjust=False).mean()
ema200 = close.ewm(span=200, adjust=False).mean()
feat['ema_distance'] = (ema50 - ema200) / close

# ── SLICE TO TARGET START (removes EMA warmup period) ────────────────
feat = feat[feat.index >= TARGET_START]
feat = feat.dropna()

# Slice raw to match feat (needed by labeler in Cell 4)
raw = raw.reindex(feat.index)

print(f'\n✅ Features computed. Shape: {feat.shape}')
print(f'   Columns: {list(feat.columns)}')
print(f'   Date range: {feat.index[0].date()} → {feat.index[-1].date()}')
feat.tail(3)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — LABELING + ALIGNMENT HELPERS                           ║
# ╚══════════════════════════════════════════════════════════════════╝

LAMBDA_CANDIDATES = [0.50, 0.75, 1.00, 1.25, 1.50]

BASE_FEATURE_COLUMNS = [
    'log_ret_close',
    'log_ret_open',
    'log_ret_high',
    'log_ret_low',
    'rolling_vol_20',
    'rsi_14',
    'macd_norm',
]

EXTRA_FEATURES = {
    'log_vix': 'log_vix',
    'vol_shock': 'vol_shock_20',
    'momentum': 'momentum_5d',
    'trend': 'ema_distance',
}

FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + [
    EXTRA_FEATURES['log_vix'],
    EXTRA_FEATURES['vol_shock'],
    EXTRA_FEATURES['momentum'],
    EXTRA_FEATURES['trend'],
]

FEATURE_GROUPS = {
    'Base price+tech (7)': BASE_FEATURE_COLUMNS.copy(),
    'Base + vol_shock (8)': BASE_FEATURE_COLUMNS + [EXTRA_FEATURES['vol_shock']],
    'Base + momentum (8)': BASE_FEATURE_COLUMNS + [EXTRA_FEATURES['momentum']],
    'Base + trend (8)': BASE_FEATURE_COLUMNS + [EXTRA_FEATURES['trend']],
    'Base + Macro/VIX (8)': BASE_FEATURE_COLUMNS + [EXTRA_FEATURES['log_vix']],
    'Base + vol/mom/trend (10, no VIX)': BASE_FEATURE_COLUMNS + [
        EXTRA_FEATURES['vol_shock'], EXTRA_FEATURES['momentum'], EXTRA_FEATURES['trend']
    ],
    'Full 11 features': FEATURE_COLUMNS.copy(),
}

FROZEN_FEATURE_GROUP_NAME = 'Base price+tech (7)'
ENSEMBLE_MODEL_NAME = 'Ensemble (Prop+LSTM+XGB)'
CLASS_NAMES = ['Down', 'Neutral', 'Up']


def generate_labels(close_series, index, lambda_val, horizon=5, vol_window=30):
    close_series = pd.Series(close_series, index=index).astype(float)
    future_ret = np.log(close_series.shift(-horizon) / close_series)
    past_ret = np.log(close_series / close_series.shift(1))
    sigma_t = past_ret.rolling(window=vol_window, min_periods=vol_window).std()
    epsilon = lambda_val * sigma_t

    labels = pd.Series(np.nan, index=index, dtype=float)
    labels[future_ret > epsilon] = 2
    labels[future_ret < -epsilon] = 0
    neutral_mask = (future_ret >= -epsilon) & (future_ret <= epsilon)
    labels[neutral_mask] = 1
    return labels


def class_distribution(labels):
    valid = pd.Series(labels).dropna()
    total = len(valid)
    counts = valid.value_counts().sort_index()
    out = {}
    for cls in [0, 1, 2]:
        out[cls] = counts.get(cls, 0) / total if total else 0.0
    return out


def distribution_balance_score(dist):
    target = np.array([1/3, 1/3, 1/3], dtype=float)
    observed = np.array([dist[0], dist[1], dist[2]], dtype=float)
    return np.abs(observed - target).sum()


def select_lambda_for_fold(close_series, feature_index, lambda_candidates, horizon=5, vol_window=30):
    rows = []
    best_lambda, best_score = None, float('inf')
    for lam in lambda_candidates:
        labels = generate_labels(close_series, feature_index, lam, horizon, vol_window)
        dist = class_distribution(labels)
        score = distribution_balance_score(dist)
        rows.append({
            'lambda': lam,
            'down_pct': dist[0] * 100.0,
            'neutral_pct': dist[1] * 100.0,
            'up_pct': dist[2] * 100.0,
            'balance_score': score,
        })
        if score < best_score:
            best_score, best_lambda = score, lam
    table = pd.DataFrame(rows).sort_values(['balance_score', 'lambda']).reset_index(drop=True)
    return best_lambda, table


def build_sequences(features_df, labels_series, lookback=60):
    feat_arr = features_df.values
    label_arr = labels_series.reindex(features_df.index).values
    date_arr = features_df.index.values

    X_list, y_list, d_list = [], [], []
    for end_idx in range(lookback - 1, len(features_df)):
        lbl = label_arr[end_idx]
        if np.isnan(lbl):
            continue
        start_idx = end_idx - lookback + 1
        window = feat_arr[start_idx:end_idx + 1]
        X_list.append(window.T)
        y_list.append(int(lbl))
        d_list.append(date_arr[end_idx])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    d = pd.to_datetime(np.array(d_list))
    return X, y, d


def normalize_fold(X_train, X_test):
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True)
    std = np.where(std < 1e-8, 1e-8, std)
    return (X_train - mean) / std, (X_test - mean) / std, mean, std


def inspect_alignment(features_df, raw_df, labels_series, lookback, horizon, sample_number=0):
    X_tmp, y_tmp, d_tmp = build_sequences(features_df, labels_series, lookback)
    anchor_date = pd.Timestamp(d_tmp[sample_number])
    start_date = features_df.index[features_df.index.get_loc(anchor_date) - lookback + 1]
    future_date = raw_df.index[raw_df.index.get_loc(anchor_date) + horizon]
    return {
        'window_start': start_date.date(),
        'window_end': anchor_date.date(),
        'target_horizon_end': future_date.date(),
        'label': int(y_tmp[sample_number]),
        'n_sequences': len(X_tmp),
    }


def evaluate_predictions(y_true, y_pred):
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    pred_counts = np.bincount(y_pred, minlength=3)
    true_counts = np.bincount(y_true, minlength=3)
    return {
        'macro_f1': macro_f1,
        'mcc': mcc,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'class_f1': f1,
        'support': support,
        'down_recall': recall[0],
        'conf_matrix': cm,
        'pred_counts': pred_counts,
        'true_counts': true_counts,
        'y_pred': y_pred,
    }


labels_preview = generate_labels(raw['Close'], feat.index, lambda_val=1.0, horizon=HORIZON, vol_window=VOL_WINDOW)
alignment_preview = inspect_alignment(feat, raw, labels_preview, LOOKBACK, HORIZON, sample_number=0)
print('Alignment preview:')
for k, v in alignment_preview.items():
    print(f'  {k}: {v}')
print('Feature groups defined:')
for group_name, cols in FEATURE_GROUPS.items():
    marker = '  [frozen final]' if group_name == FROZEN_FEATURE_GROUP_NAME else ''
    print(f'  {group_name}: {len(cols)} columns{marker}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FOLD SPECIFICATION + MATERIALIZATION                   ║
# ╚══════════════════════════════════════════════════════════════════╝

SAMPLE_END_DATES = feat.index[LOOKBACK - 1 : len(feat) - HORIZON]
TOTAL_SAMPLES = len(SAMPLE_END_DATES)
TEST_SIZE = int(TOTAL_SAMPLES * TEST_RATIO)
TRAIN_END0 = TOTAL_SAMPLES - N_FOLDS * TEST_SIZE

print('=' * 78)
print('  SAMPLE CALENDAR')
print('=' * 78)
print(f'Total candidate aligned samples: {TOTAL_SAMPLES}')
print(f'Lookback={LOOKBACK}, Horizon={HORIZON}, Test size per fold={TEST_SIZE}')
print(f'First sample end-date: {SAMPLE_END_DATES[0].date()}')
print(f'Last sample end-date : {SAMPLE_END_DATES[-1].date()}')
print(f'Frozen final feature group: {FROZEN_FEATURE_GROUP_NAME}')


def make_fold_specs(features_df, raw_df, sample_dates, n_folds=3, test_size=100):
    fold_specs = []
    print('\n' + '=' * 78)
    print('  WALK-FORWARD SPECIFICATION  (fold-local lambda selection)')
    print('=' * 78)
    for fid in range(1, n_folds + 1):
        ts = TRAIN_END0 + (fid - 1) * test_size
        te = ts + test_size
        train_end_date = pd.Timestamp(sample_dates[ts - 1])
        test_start_date = pd.Timestamp(sample_dates[ts])
        test_end_date = pd.Timestamp(sample_dates[te - 1])

        train_feature_index = features_df.loc[:train_end_date].index
        best_lambda, lambda_table = select_lambda_for_fold(
            raw_df.loc[train_feature_index, 'Close'],
            train_feature_index,
            lambda_candidates=LAMBDA_CANDIDATES,
            horizon=HORIZON,
            vol_window=VOL_WINDOW,
        )

        fold_specs.append({
            'fold_id': fid,
            'train_sample_end': ts,
            'test_sample_start': ts,
            'test_sample_end': te,
            'train_end_date': train_end_date,
            'test_start_date': test_start_date,
            'test_end_date': test_end_date,
            'lambda': best_lambda,
            'lambda_table': lambda_table,
        })

        print(f'Fold {fid}: Train {sample_dates[0].date()} -> {train_end_date.date()} ({ts} samples)')
        print(f'         Test  {test_start_date.date()} -> {test_end_date.date()} ({test_size} samples)')
        print(f'         Selected lambda: {best_lambda:.2f}')
        display_cols = ['lambda', 'down_pct', 'neutral_pct', 'up_pct', 'balance_score']
        print(lambda_table[display_cols].to_string(index=False, float_format=lambda x: f'{x:0.3f}'))
        print('-' * 78)
    return fold_specs


def materialize_fold_dataset(features_df, raw_df, fold_spec, feature_cols):
    labels_all = generate_labels(
        raw_df['Close'], features_df.index, fold_spec['lambda'], HORIZON, VOL_WINDOW
    )
    X_all, y_all, dates_all = build_sequences(features_df[feature_cols], labels_all, LOOKBACK)

    assert len(dates_all) == TOTAL_SAMPLES, 'Aligned sample count drifted unexpectedly.'
    assert pd.Index(dates_all).equals(pd.Index(SAMPLE_END_DATES)), 'Sample date alignment mismatch.'

    ts = fold_spec['test_sample_start']
    te = fold_spec['test_sample_end']

    X_train_raw, y_train = X_all[:ts], y_all[:ts]
    X_test_raw, y_test = X_all[ts:te], y_all[ts:te]
    dates_train, dates_test = dates_all[:ts], dates_all[ts:te]

    X_train, X_test, mean, std = normalize_fold(X_train_raw, X_test_raw)

    vol_feature_idx = feature_cols.index('rolling_vol_20')
    regime_signal = X_test_raw[:, vol_feature_idx, -1]
    med = np.median(regime_signal)
    low_mask = regime_signal <= med
    high_mask = regime_signal > med
    regime_valid = (low_mask.sum() >= 50 and high_mask.sum() >= 50)

    train_dist = np.bincount(y_train, minlength=3)
    test_dist = np.bincount(y_test, minlength=3)

    return {
        'fold_id': fold_spec['fold_id'],
        'lambda': fold_spec['lambda'],
        'feature_cols': feature_cols,
        'feature_group_name': None,
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test,
        'X_train_raw': X_train_raw,
        'X_test_raw': X_test_raw,
        'dates_train': dates_train,
        'dates_test': dates_test,
        'norm_mean': mean,
        'norm_std': std,
        'low_vol_mask': low_mask,
        'high_vol_mask': high_mask,
        'regime_valid': regime_valid,
        'train_class_counts': train_dist,
        'test_class_counts': test_dist,
        'lambda_table': fold_spec['lambda_table'],
    }


def build_feature_group_folds(group_name, feature_cols):
    group_folds = []
    for spec in FOLD_SPECS:
        fold = materialize_fold_dataset(feat, raw, spec, feature_cols)
        fold['feature_group_name'] = group_name
        group_folds.append(fold)
    return group_folds


FOLD_SPECS = make_fold_specs(feat, raw, SAMPLE_END_DATES, N_FOLDS, TEST_SIZE)
SELECTED_FEATURE_GROUP_NAME = FROZEN_FEATURE_GROUP_NAME
SELECTED_FEATURE_COLUMNS = FEATURE_GROUPS[SELECTED_FEATURE_GROUP_NAME]
SELECTED_FOLDS = build_feature_group_folds(SELECTED_FEATURE_GROUP_NAME, SELECTED_FEATURE_COLUMNS)

print('\n' + '=' * 78)
print(f'  FROZEN FINAL FOLD DIAGNOSTICS -- {SELECTED_FEATURE_GROUP_NAME}')
print('=' * 78)
for fold in SELECTED_FOLDS:
    d0 = pd.Timestamp(fold['dates_test'][0]).date()
    d1 = pd.Timestamp(fold['dates_test'][-1]).date()
    print(f'Fold {fold["fold_id"]}: test {d0} -> {d1}')
    print(f'  lambda={fold["lambda"]:.2f}')
    print(f'  train class counts={fold["train_class_counts"].tolist()}')
    print(f'  test  class counts={fold["test_class_counts"].tolist()}')
    print(f'  regime valid={fold["regime_valid"]} (low={int(fold["low_vol_mask"].sum())}, high={int(fold["high_vol_mask"].sum())})')
    print('-' * 78)


---
## Section 2 -- Models

The proposed model is rebuilt to be small and stable for this dataset:
- GroupNorm instead of BatchNorm inside the temporal encoder
- one lightweight Transformer block
- mean pooling over time for the attention model
- shared training stack across all neural models

The attention model is compared against:
- a CNN-only ablation
- an LSTM baseline
- XGBoost
- Dummy classifier
---


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — MODEL DEFINITIONS                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

MODEL_DROPOUT = 0.15
MODEL_WIDTH = 48


def _num_groups(channels):
    for g in [8, 6, 4, 3, 2]:
        if channels % g == 0:
            return g
    return 1


class ConvBlock1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dropout=0.15):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad)
        self.norm = nn.GroupNorm(_num_groups(out_ch), out_ch)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(self.norm(self.conv(x))))


class ResidualConvBlock(nn.Module):
    def __init__(self, channels, kernel_size=3, dropout=0.15):
        super().__init__()
        self.block1 = ConvBlock1d(channels, channels, kernel_size, dropout)
        self.block2 = ConvBlock1d(channels, channels, kernel_size, dropout)

    def forward(self, x):
        return x + self.block2(self.block1(x))


class TemporalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ProposedModel(nn.Module):
    def __init__(self, C=11, T=60, width=48, dropout=0.15, n_classes=3):
        super().__init__()
        self.stem = nn.Conv1d(C, width, kernel_size=1)
        self.conv = nn.Sequential(
            ConvBlock1d(width, width, kernel_size=5, dropout=dropout),
            ResidualConvBlock(width, kernel_size=3, dropout=dropout),
            ResidualConvBlock(width, kernel_size=3, dropout=dropout),
        )
        self.pos_enc = TemporalPositionalEncoding(width, max_len=T + 4)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=width,
            nhead=4,
            dim_feedforward=width * 2,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes),
        )
        self._attn_weights = None

    def forward(self, x, return_attn=False):
        x = self.stem(x)
        x = self.conv(x)
        x = x.transpose(1, 2)
        x = self.pos_enc(x)

        if return_attn:
            layer = self.transformer.layers[0]
            x_norm = layer.norm1(x)
            _, attn = layer.self_attn(
                x_norm, x_norm, x_norm,
                need_weights=True,
                average_attn_weights=True,
            )
            self._attn_weights = attn.detach().cpu().numpy()

        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.head(x)


class CNNBaseline(nn.Module):
    def __init__(self, C=11, width=48, dropout=0.15, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(C, width, kernel_size=1),
            ConvBlock1d(width, width, kernel_size=5, dropout=dropout),
            ResidualConvBlock(width, kernel_size=3, dropout=dropout),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.LayerNorm(width),
            nn.Linear(width, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class LSTMBaseline(nn.Module):
    def __init__(self, C=11, hidden=96, dropout=0.15, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=C,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            dropout=0.0,
            bidirectional=False,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 48),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(48, n_classes),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


C_FEAT = len(SELECTED_FEATURE_COLUMNS)
T_FEAT = LOOKBACK
print(f'Frozen final feature group: {SELECTED_FEATURE_GROUP_NAME}')
print(f'Input: C_FEAT={C_FEAT}, T_FEAT={T_FEAT}')
print(f'Proposed params: {count_trainable_params(ProposedModel(C=C_FEAT, T=T_FEAT)):,}')
print(f'CNN params     : {count_trainable_params(CNNBaseline(C=C_FEAT)):,}')
print(f'LSTM params    : {count_trainable_params(LSTMBaseline(C=C_FEAT)):,}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — FROZEN FINAL TRAINING + REPORTING                     ║
# ╚══════════════════════════════════════════════════════════════════╝

EPOCHS = 120
BATCH_SIZE = 64
LR = 2e-4
PATIENCE = 20
VAL_SPLIT = 0.15
WEIGHT_DECAY = 3e-4
RUN_MULTI_SEED = True
MULTI_SEED_LIST = [7, 21, 42, 84, 168]
RUN_ENSEMBLE = True

FINAL_MODEL_ORDER = [
    'Proposed (CNN+Attn)',
    'LSTM',
    'XGBoost',
    '1D-CNN (no attn)',
    'Dummy',
    ENSEMBLE_MODEL_NAME,
]


def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def temporal_train_val_split(X_train, y_train, val_split=0.15):
    n_val = max(1, int(len(X_train) * val_split))
    n_val = min(n_val, len(X_train) - 1)
    return X_train[:-n_val], y_train[:-n_val], X_train[-n_val:], y_train[-n_val:]


def compute_class_weights(y_train, max_weight=1.8):
    counts = np.bincount(y_train, minlength=3).astype(float)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    weights = np.clip(weights, 0.75, max_weight)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def build_loss(loss_name, y_train):
    if loss_name == 'ce':
        return nn.CrossEntropyLoss()
    if loss_name == 'weighted_ce':
        return nn.CrossEntropyLoss(weight=compute_class_weights(y_train))
    raise ValueError(f'Unsupported loss: {loss_name}')


def predict_from_logits(logits):
    return logits.argmax(axis=1)


def predict_torch_logits(model, X):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=BATCH_SIZE)
    logits = []
    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(DEVICE)
            logits.append(model(xb).cpu().numpy())
    return np.concatenate(logits, axis=0)


def train_torch_model(model_ctor, fold_data, model_name='Model', loss_name='weighted_ce', seed=None):
    seed = SEED + fold_data['fold_id'] if seed is None else seed
    set_all_seeds(seed)
    X_train, y_train = fold_data['X_train'], fold_data['y_train']
    X_tr, y_tr, X_vl, y_vl = temporal_train_val_split(X_train, y_train, VAL_SPLIT)

    model = model_ctor().to(DEVICE)
    criterion = build_loss(loss_name, y_tr)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-5)

    tr_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.int64))
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)

    best_state = None
    best_stats = {'macro_f1': -1.0, 'mcc': -2.0, 'epoch': 0}
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in tr_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        val_logits = predict_torch_logits(model, X_vl)
        val_pred = predict_from_logits(val_logits)
        val_f1 = f1_score(y_vl, val_pred, average='macro', zero_division=0)
        val_mcc = matthews_corrcoef(y_vl, val_pred)
        scheduler.step(val_f1)

        improved = (val_f1 > best_stats['macro_f1']) or (
            np.isclose(val_f1, best_stats['macro_f1']) and val_mcc > best_stats['mcc']
        )
        if improved:
            best_state = copy.deepcopy(model.state_dict())
            best_stats = {'macro_f1': val_f1, 'mcc': val_mcc, 'epoch': epoch}
            wait = 0
        else:
            wait += 1

        if epoch % 20 == 0 or epoch == 1:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'    [{model_name}] epoch={epoch:3d}  val_macro_f1={val_f1:.4f}  val_mcc={val_mcc:.4f}  lr={lr_now:.2e}')

        if wait >= PATIENCE:
            print(f'    [{model_name}] Early stop @ epoch {epoch}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_logits = predict_torch_logits(model, fold_data['X_test'])
    y_pred = predict_from_logits(test_logits)
    metrics = evaluate_predictions(fold_data['y_test'], y_pred)
    metrics.update({
        'model_name': model_name,
        'loss_name': loss_name,
        'best_val_macro_f1': best_stats['macro_f1'],
        'best_val_mcc': best_stats['mcc'],
        'best_epoch': best_stats['epoch'],
        'trained_model': model,
        'test_logits': test_logits,
        'seed': seed,
    })
    return metrics


def train_xgboost_model(fold_data, model_name='XGBoost', seed=None):
    seed = SEED + fold_data['fold_id'] if seed is None else seed
    X_train, y_train = fold_data['X_train'], fold_data['y_train']
    X_tr, y_tr, X_vl, y_vl = temporal_train_val_split(X_train, y_train, VAL_SPLIT)
    X_tr = X_tr.reshape(len(X_tr), -1)
    X_te = fold_data['X_test'].reshape(len(fold_data['X_test']), -1)

    model = XGBClassifier(
        n_estimators=450,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=4,
        reg_alpha=0.2,
        reg_lambda=1.2,
        objective='multi:softprob',
        num_class=3,
        random_state=seed,
        n_jobs=-1,
        verbosity=0,
        eval_metric='mlogloss',
    )
    model.fit(X_tr, y_tr)
    prob = model.predict_proba(X_te)
    logits = np.log(np.clip(prob, 1e-9, 1.0))
    y_pred = predict_from_logits(logits)
    metrics = evaluate_predictions(fold_data['y_test'], y_pred)
    metrics.update({
        'model_name': model_name,
        'loss_name': 'tree',
        'best_val_macro_f1': None,
        'best_val_mcc': None,
        'best_epoch': None,
        'trained_model': model,
        'test_logits': logits,
        'seed': seed,
    })
    return metrics


def train_dummy_model(fold_data, model_name='Dummy'):
    dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
    dummy.fit(fold_data['X_train'].reshape(len(fold_data['X_train']), -1), fold_data['y_train'])
    y_pred = dummy.predict(fold_data['X_test'].reshape(len(fold_data['X_test']), -1))
    metrics = evaluate_predictions(fold_data['y_test'], y_pred)
    metrics.update({
        'model_name': model_name,
        'loss_name': 'dummy',
        'best_val_macro_f1': None,
        'best_val_mcc': None,
        'best_epoch': None,
        'trained_model': dummy,
        'test_logits': None,
        'seed': None,
    })
    return metrics


def attach_regime_scores(result, fold_data):
    if not fold_data['regime_valid']:
        result['lv_f1'] = None
        result['hv_f1'] = None
        return result
    lv = fold_data['low_vol_mask']
    hv = fold_data['high_vol_mask']
    y_test = fold_data['y_test']
    y_pred = result['y_pred']
    result['lv_f1'] = f1_score(y_test[lv], y_pred[lv], average='macro', zero_division=0)
    result['hv_f1'] = f1_score(y_test[hv], y_pred[hv], average='macro', zero_division=0)
    return result


def build_model_specs(C_local, T_local):
    return [
        ('Proposed (CNN+Attn)', lambda: ProposedModel(C=C_local, T=T_local), 'weighted_ce', 'torch'),
        ('1D-CNN (no attn)', lambda: CNNBaseline(C=C_local), 'weighted_ce', 'torch'),
        ('LSTM', lambda: LSTMBaseline(C=C_local), 'weighted_ce', 'torch'),
        ('XGBoost', None, None, 'xgb'),
        ('Dummy', None, None, 'dummy'),
    ]


def run_main_experiment(folds):
    all_results = {}

    for fold in folds:
        fid = fold['fold_id']
        te0 = pd.Timestamp(fold['dates_test'][0]).date()
        te1 = pd.Timestamp(fold['dates_test'][-1]).date()
        print('\n' + '=' * 78)
        print(f'  FOLD {fid} | Test: {te0} -> {te1} | lambda={fold["lambda"]:.2f} | features={fold["feature_group_name"]}')
        print('=' * 78)

        C_local = len(fold['feature_cols'])
        T_local = fold['X_train'].shape[2]
        model_specs = build_model_specs(C_local, T_local)

        for name, ctor, loss_name, family in model_specs:
            print(f'\n  Training: {name}')
            if family == 'torch':
                result = train_torch_model(ctor, fold, name, loss_name=loss_name)
            elif family == 'xgb':
                result = train_xgboost_model(fold, name)
            else:
                result = train_dummy_model(fold, name)

            result = attach_regime_scores(result, fold)
            result['fold_id'] = fid
            result['test_start'] = te0
            result['test_end'] = te1
            result['lambda'] = fold['lambda']
            result['feature_group_name'] = fold['feature_group_name']
            all_results.setdefault(name, []).append(result)

            print(f'    Macro F1={result["macro_f1"]:.4f}  MCC={result["mcc"]:.4f}  Down-recall={result["down_recall"]:.4f}')
            print(f'    Pred counts={result["pred_counts"].tolist()}  True counts={result["true_counts"].tolist()}')

            if name == 'Proposed (CNN+Attn)':
                fold['trained_proposed'] = result['trained_model']
                fold['trained_proposed_logits'] = result['test_logits']
                fold['trained_proposed_pred'] = result['y_pred']

        if RUN_ENSEMBLE:
            ensemble_names = ['Proposed (CNN+Attn)', 'LSTM', 'XGBoost']
            logits_stack = [all_results[name][-1]['test_logits'] for name in ensemble_names]
            ensemble_logits = sum(logits_stack) / len(logits_stack)
            ensemble_pred = predict_from_logits(ensemble_logits)
            ensemble_result = evaluate_predictions(fold['y_test'], ensemble_pred)
            ensemble_result.update({
                'model_name': ENSEMBLE_MODEL_NAME,
                'loss_name': 'ensemble',
                'best_val_macro_f1': None,
                'best_val_mcc': None,
                'best_epoch': None,
                'trained_model': None,
                'test_logits': ensemble_logits,
                'seed': None,
                'fold_id': fid,
                'test_start': te0,
                'test_end': te1,
                'lambda': fold['lambda'],
                'feature_group_name': fold['feature_group_name'],
            })
            ensemble_result = attach_regime_scores(ensemble_result, fold)
            all_results.setdefault(ENSEMBLE_MODEL_NAME, []).append(ensemble_result)
            print('\n  Ensemble: Proposed + LSTM + XGBoost')
            print(f'    Macro F1={ensemble_result["macro_f1"]:.4f}  MCC={ensemble_result["mcc"]:.4f}  Down-recall={ensemble_result["down_recall"]:.4f}')
            print(f'    Pred counts={ensemble_result["pred_counts"].tolist()}  True counts={ensemble_result["true_counts"].tolist()}')

    return all_results


def summarize_results(all_results):
    rows = []
    for model_name, fold_results in all_results.items():
        f1s = [r['macro_f1'] for r in fold_results]
        mccs = [r['mcc'] for r in fold_results]
        accs = [r['accuracy'] for r in fold_results]
        drs = [r['down_recall'] for r in fold_results]
        rows.append({
            'Model': model_name,
            'Mean F1': np.mean(f1s),
            'Std F1': np.std(f1s),
            'Mean MCC': np.mean(mccs),
            'Std MCC': np.std(mccs),
            'Mean Acc': np.mean(accs),
            'Mean DownRec': np.mean(drs),
        })
    rows = sorted(rows, key=lambda r: (r['Mean F1'], r['Mean MCC']), reverse=True)
    return pd.DataFrame(rows)


def build_ensemble_class_tables(all_results):
    fold_rows = []
    for res in all_results[ENSEMBLE_MODEL_NAME]:
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            fold_rows.append({
                'Fold': res['fold_id'],
                'Class': cls_name,
                'Precision': res['precision'][cls_idx],
                'Recall': res['recall'][cls_idx],
                'F1': res['class_f1'][cls_idx],
                'Support': int(res['support'][cls_idx]),
            })
    fold_df = pd.DataFrame(fold_rows)
    mean_df = (
        fold_df.groupby('Class', as_index=False)
        .agg({
            'Precision': 'mean',
            'Recall': 'mean',
            'F1': 'mean',
            'Support': 'sum',
        })
        .rename(columns={'Support': 'Total Support'})
    )
    return fold_df, mean_df


def build_confusion_comparison_df(all_results, proposed_name='Proposed (CNN+Attn)'):
    rows = []
    for prop_res, ens_res in zip(all_results[proposed_name], all_results[ENSEMBLE_MODEL_NAME]):
        prop_cm = prop_res['conf_matrix'].astype(float)
        ens_cm = ens_res['conf_matrix'].astype(float)
        prop_pct = prop_cm / np.where(prop_cm.sum(axis=1, keepdims=True) == 0, 1, prop_cm.sum(axis=1, keepdims=True)) * 100.0
        ens_pct = ens_cm / np.where(ens_cm.sum(axis=1, keepdims=True) == 0, 1, ens_cm.sum(axis=1, keepdims=True)) * 100.0

        best = None
        for actual_idx, actual_name in enumerate(CLASS_NAMES):
            for pred_idx, pred_name in enumerate(CLASS_NAMES):
                if actual_idx == pred_idx:
                    continue
                delta = prop_pct[actual_idx, pred_idx] - ens_pct[actual_idx, pred_idx]
                if best is None or delta > best['reduction_pp']:
                    best = {
                        'Fold': prop_res['fold_id'],
                        'Actual': actual_name,
                        'Predicted': pred_name,
                        'Prop %': prop_pct[actual_idx, pred_idx],
                        'Ensemble %': ens_pct[actual_idx, pred_idx],
                        'reduction_pp': delta,
                    }
        rows.append(best)
    return pd.DataFrame(rows)


def run_multi_seed_robustness(seed_list, feature_group_name, feature_cols):
    records = []
    for seed in seed_list:
        print('\n' + '=' * 78)
        print(f'MULTI-SEED ROBUSTNESS RUN -- seed={seed} -- {feature_group_name}')
        print('=' * 78)
        seed_folds = build_feature_group_folds(feature_group_name, feature_cols)
        seed_results = {}
        for fold in seed_folds:
            C_local = len(fold['feature_cols'])
            T_local = fold['X_train'].shape[2]
            for name, ctor, loss_name, family in build_model_specs(C_local, T_local):
                if family == 'torch':
                    result = train_torch_model(ctor, fold, name, loss_name=loss_name, seed=seed + fold['fold_id'])
                elif family == 'xgb':
                    result = train_xgboost_model(fold, name, seed=seed + fold['fold_id'])
                else:
                    result = train_dummy_model(fold, name)
                result['fold_id'] = fold['fold_id']
                seed_results.setdefault(name, []).append(result)
                records.append({
                    'seed': seed,
                    'fold_id': fold['fold_id'],
                    'model': name,
                    'macro_f1': result['macro_f1'],
                    'mcc': result['mcc'],
                    'accuracy': result['accuracy'],
                    'down_recall': result['down_recall'],
                })

            ensemble_logits = sum(seed_results[name][-1]['test_logits'] for name in ['Proposed (CNN+Attn)', 'LSTM', 'XGBoost']) / 3.0
            ensemble_pred = predict_from_logits(ensemble_logits)
            ensemble_result = evaluate_predictions(fold['y_test'], ensemble_pred)
            records.append({
                'seed': seed,
                'fold_id': fold['fold_id'],
                'model': ENSEMBLE_MODEL_NAME,
                'macro_f1': ensemble_result['macro_f1'],
                'mcc': ensemble_result['mcc'],
                'accuracy': ensemble_result['accuracy'],
                'down_recall': ensemble_result['down_recall'],
            })

    fold_df = pd.DataFrame(records)
    per_seed_df = (
        fold_df.groupby(['seed', 'model'], as_index=False)
        .agg({
            'macro_f1': 'mean',
            'mcc': 'mean',
            'accuracy': 'mean',
            'down_recall': 'mean',
        })
        .sort_values(['model', 'seed'])
        .reset_index(drop=True)
    )
    summary_df = (
        per_seed_df.groupby('model', as_index=False)
        .agg({
            'macro_f1': ['mean', 'std'],
            'mcc': ['mean', 'std'],
            'accuracy': ['mean', 'std'],
            'down_recall': ['mean', 'std'],
        })
    )
    summary_df.columns = [
        'Model',
        'Mean F1', 'Std F1',
        'Mean MCC', 'Std MCC',
        'Mean Acc', 'Std Acc',
        'Mean DownRec', 'Std DownRec',
    ]
    summary_df = summary_df.sort_values(['Mean F1', 'Mean MCC'], ascending=False).reset_index(drop=True)
    return fold_df, per_seed_df, summary_df


FINAL_RESULTS = run_main_experiment(SELECTED_FOLDS)
SUMMARY_DF = summarize_results(FINAL_RESULTS)
BEST_MODEL_NAME = SUMMARY_DF.iloc[0]['Model']
BEST_PROPOSED_NAME = 'Proposed (CNN+Attn)'
ENSEMBLE_PER_FOLD_CLASS_DF, ENSEMBLE_MEAN_CLASS_DF = build_ensemble_class_tables(FINAL_RESULTS)
CONFUSION_COMPARISON_DF = build_confusion_comparison_df(FINAL_RESULTS, BEST_PROPOSED_NAME)

MULTI_SEED_FOLD_DF, MULTI_SEED_PER_SEED_DF, MULTI_SEED_SUMMARY_DF = (None, None, None)
if RUN_MULTI_SEED:
    MULTI_SEED_FOLD_DF, MULTI_SEED_PER_SEED_DF, MULTI_SEED_SUMMARY_DF = run_multi_seed_robustness(
        MULTI_SEED_LIST,
        SELECTED_FEATURE_GROUP_NAME,
        SELECTED_FEATURE_COLUMNS,
    )


---
## Section 3 -- Reporting and Diagnostics

This section is a **reporting-only freeze pass**.
The model stack is fixed to the final accepted systems and no new tuning is performed.

Fold 1 note for the viva:
- Fold 1 spans **February 2020 to May 2021**.
- This period includes the COVID crash and the extraordinary policy-driven recovery.
- We do **not** retune the architecture to fit this anomaly, because doing so would overfit an exogenous macro shock rather than improve the underlying technical-indicator model.
---


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — RESULTS TABLES + DIAGNOSTICS                          ║
# ╚══════════════════════════════════════════════════════════════════╝

print('\n' + '=' * 98)
print('  FROZEN FINAL RESULTS -- MEAN +/- STD ACROSS ALL FOLDS')
print('=' * 98)
print(f'{"Model":<42} {"Macro F1":>16} {"MCC":>16} {"Accuracy":>10} {"Down Rec":>10}')
print('-' * 98)
for _, r in SUMMARY_DF.iterrows():
    print(f"{r['Model']:<42} {r['Mean F1']:.4f}+/-{r['Std F1']:.3f}  {r['Mean MCC']:.4f}+/-{r['Std MCC']:.3f}  {r['Mean Acc']:.4f}    {r['Mean DownRec']:.4f}")
print('=' * 98)

print('\nPer-fold diagnostic snapshots:')
for model_name in FINAL_MODEL_ORDER:
    fold_results = FINAL_RESULTS[model_name]
    print('\n' + model_name)
    for r in fold_results:
        print(
            f"  Fold {r['fold_id']}: F1={r['macro_f1']:.4f} MCC={r['mcc']:.4f} "
            f"pred={r['pred_counts'].tolist()} true={r['true_counts'].tolist()} "
            f"best_epoch={r['best_epoch']}"
        )

print('\n' + '=' * 98)
print('  ENSEMBLE PER-CLASS METRICS BY FOLD')
print('=' * 98)
print(ENSEMBLE_PER_FOLD_CLASS_DF.to_string(index=False, float_format=lambda x: f'{x:0.4f}'))
print('=' * 98)

print('\n' + '=' * 98)
print('  ENSEMBLE MEAN PER-CLASS METRICS ACROSS FOLDS')
print('=' * 98)
print(ENSEMBLE_MEAN_CLASS_DF.to_string(index=False, float_format=lambda x: f'{x:0.4f}'))
print('=' * 98)

print('\n' + '=' * 98)
print('  CONFUSION-MATRIX IMPROVEMENT SNAPSHOT (ENSEMBLE VS PROPOSED)')
print('=' * 98)
for _, row in CONFUSION_COMPARISON_DF.iterrows():
    if row['reduction_pp'] > 0:
        print(
            f"Fold {int(row['Fold'])}: Ensemble reduced {row['Actual']} -> {row['Predicted']} confusion "
            f"from {row['Prop %']:.1f}% to {row['Ensemble %']:.1f}% ({row['reduction_pp']:.1f} pp)."
        )
    else:
        print(
            f"Fold {int(row['Fold'])}: Ensemble did not reduce the worst off-diagonal confusion relative to the proposed model."
        )
print('=' * 98)

print(f'\nBest mean model: {BEST_MODEL_NAME}')
print(f'Frozen standalone proposed model: {BEST_PROPOSED_NAME}')
print(f'Frozen final feature group: {SELECTED_FEATURE_GROUP_NAME}')

if MULTI_SEED_SUMMARY_DF is not None:
    print('\n' + '=' * 98)
    print('  ROBUSTNESS ACROSS RANDOM SEEDS -- SUMMARY')
    print('=' * 98)
    print(MULTI_SEED_SUMMARY_DF.to_string(index=False, float_format=lambda x: f'{x:0.4f}'))
    print('=' * 98)

    print('\n' + '=' * 98)
    print('  ROBUSTNESS ACROSS RANDOM SEEDS -- PER-SEED APPENDIX')
    print('=' * 98)
    print(MULTI_SEED_PER_SEED_DF.to_string(index=False, float_format=lambda x: f'{x:0.4f}'))
    print('=' * 98)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — VISUALIZATIONS                                         ║
# ╚══════════════════════════════════════════════════════════════════╝

MODEL_COLORS = {
    'Proposed (CNN+Attn)': '#0A7B83',
    '1D-CNN (no attn)': '#E8A020',
    'LSTM': '#6B5EA8',
    'XGBoost': '#2E9E60',
    ENSEMBLE_MODEL_NAME: '#8C564B',
    'Dummy': '#888888',
}

prop_key = BEST_PROPOSED_NAME
ensemble_key = ENSEMBLE_MODEL_NAME
plot_title_suffix = f'{SELECTED_FEATURE_GROUP_NAME}'

fig, axes = plt.subplots(2, N_FOLDS, figsize=(5.6 * N_FOLDS, 8.8))
if N_FOLDS == 1:
    axes = np.array(axes).reshape(2, 1)
fig.suptitle(f'Confusion Matrices -- Proposed vs Ensemble ({plot_title_suffix})', fontsize=14, fontweight='bold', y=1.02)
for i, fold in enumerate(SELECTED_FOLDS):
    te = pd.Timestamp(fold['dates_test'][0]).date()
    for row_idx, model_name in enumerate([prop_key, ensemble_key]):
        res = FINAL_RESULTS[model_name][i]
        cm = res['conf_matrix']
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_pct = cm.astype(float) / np.where(row_sums == 0, 1, row_sums) * 100.0
        ax = axes[row_idx, i]
        sns.heatmap(
            cm_pct, annot=True, fmt='.1f', cmap='Blues', cbar=False, linewidths=0.5,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, vmin=0, vmax=100, ax=ax
        )
        model_label = 'Proposed' if model_name == prop_key else 'Ensemble'
        ax.set_title(
            f'{model_label} | Fold {fold["fold_id"]} (test from {te})\n'
            f'F1={res["macro_f1"]:.4f} MCC={res["mcc"]:.4f}',
            fontsize=10.5,
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices_comparison_final.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Model Comparison Across Folds -- {plot_title_suffix}', fontsize=14, fontweight='bold')
for metric_key, metric_label, ax in [('macro_f1', 'Macro F1', axes[0]), ('mcc', 'MCC', axes[1])]:
    for model_name in FINAL_MODEL_ORDER:
        fold_results = FINAL_RESULTS[model_name]
        vals = [r[metric_key] for r in fold_results]
        folds = [r['fold_id'] for r in fold_results]
        lw = 2.6 if model_name == prop_key else 1.6
        ls = '-' if model_name in [prop_key, ensemble_key] else '--'
        ax.plot(folds, vals, marker='o', linewidth=lw, linestyle=ls, color=MODEL_COLORS.get(model_name, 'grey'), label=model_name)
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric_label)
    ax.set_xticks(range(1, N_FOLDS + 1))
    ax.set_title(f'{metric_label} per Fold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fold_comparison_final.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Mean +/- Std Across All Folds -- {plot_title_suffix}', fontsize=14, fontweight='bold')
sorted_names = SUMMARY_DF['Model'].tolist()
colors = [MODEL_COLORS.get(name, 'grey') for name in sorted_names]
for metric_key, std_key, label, ax in [('Mean F1', 'Std F1', 'Mean Macro F1', axes[0]), ('Mean MCC', 'Std MCC', 'Mean MCC', axes[1])]:
    means = SUMMARY_DF[metric_key].tolist()
    stds = SUMMARY_DF[std_key].tolist()
    bars = ax.barh(sorted_names, means, xerr=stds, color=colors, alpha=0.85, capsize=4, edgecolor='white')
    for bar, val in zip(bars, means):
        ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)
    ax.set_xlabel(label)
    ax.set_title(label)
    ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('summary_bars_final.png', dpi=150, bbox_inches='tight')
plt.show()

if MULTI_SEED_SUMMARY_DF is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Robustness Across Random Seeds -- Frozen Final Comparison', fontsize=13, fontweight='bold')
    seed_names = MULTI_SEED_SUMMARY_DF['Model'].tolist()
    seed_colors = [MODEL_COLORS.get(name, 'grey') for name in seed_names]
    for metric_key, std_key, label, ax in [('Mean F1', 'Std F1', 'Mean Macro F1', axes[0]), ('Mean MCC', 'Std MCC', 'Mean MCC', axes[1])]:
        means = MULTI_SEED_SUMMARY_DF[metric_key].tolist()
        stds = MULTI_SEED_SUMMARY_DF[std_key].tolist()
        bars = ax.barh(seed_names, means, xerr=stds, color=seed_colors, alpha=0.85, capsize=4, edgecolor='white')
        for bar, val in zip(bars, means):
            ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)
        ax.set_xlabel(label)
        ax.set_title(label)
        ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig('multi_seed_summary_final.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — ATTENTION VISUALIZATION                               ║
# ╚══════════════════════════════════════════════════════════════════╝

last_fold = SELECTED_FOLDS[-1]
trained_prop = last_fold.get('trained_proposed')

if trained_prop is None:
    print('No trained proposed model found. Run Cell 7 first.')
else:
    trained_prop.eval()
    X_te = torch.tensor(last_fold['X_test'], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        _ = trained_prop(X_te, return_attn=True)
    attn_all = trained_prop._attn_weights
    y_pred = last_fold['trained_proposed_pred']

    if attn_all is None:
        print('Attention weights not captured.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        fig.suptitle(f'Mean Attention Weights by Predicted Class -- {SELECTED_FEATURE_GROUP_NAME}', fontsize=12, fontweight='bold')
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            mask = y_pred == cls_idx
            ax = axes[cls_idx]
            if mask.sum() == 0:
                ax.set_title(f'{cls_name}\n(no samples)')
                ax.axis('off')
                continue
            mean_attn = attn_all[mask].mean(axis=0)
            im = ax.imshow(mean_attn, cmap='viridis', aspect='auto', vmin=0)
            ax.set_title(f'{cls_name}\n(n={mask.sum()} samples)', fontsize=11)
            ax.set_xlabel('Key day in lookback')
            ax.set_ylabel('Query day in lookback')
            plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        plt.savefig('attention_heatmap_final.png', dpi=150, bbox_inches='tight')
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 4))
        cls_colors = ['#D04040', '#888888', '#2E9E60']
        for cls_idx, (cls_name, col) in enumerate(zip(CLASS_NAMES, cls_colors)):
            mask = y_pred == cls_idx
            if mask.sum() == 0:
                continue
            cls_attn = attn_all[mask].mean(axis=(0, 1))
            ax.plot(range(1, LOOKBACK + 1), cls_attn, label=f'Class: {cls_name}', color=col, linewidth=2)
        ax.plot(range(1, LOOKBACK + 1), attn_all.mean(axis=(0, 1)), label='Overall mean', color='#0A7B83', linewidth=2.5, linestyle='--')
        ax.set_xlabel('Day in lookback window (1 = oldest, 60 = most recent)')
        ax.set_ylabel('Mean attention weight')
        ax.set_title('Which Historical Days Get the Most Attention?', fontweight='bold')
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig('attention_by_day_final.png', dpi=150, bbox_inches='tight')
        plt.show()

        print('Attention guide:')
        print('  Compare whether the strongest attention zones are class-specific or shared.')
        print('  If patterns remain smooth and shared, attention is helping only weakly and mostly as a context smoother.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — SAVE RESULTS TO GOOGLE DRIVE (OPTIONAL)               ║
# ╚══════════════════════════════════════════════════════════════════╝

SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    import os
    import shutil

    drive.mount('/content/drive')
    out_dir = '/content/drive/MyDrive/Kairos_v5_ReportingFreeze_Results'
    os.makedirs(out_dir, exist_ok=True)

    SUMMARY_DF.to_csv(f'{out_dir}/final_summary_results.csv', index=False)
    ENSEMBLE_PER_FOLD_CLASS_DF.to_csv(f'{out_dir}/ensemble_per_fold_class_metrics.csv', index=False)
    ENSEMBLE_MEAN_CLASS_DF.to_csv(f'{out_dir}/ensemble_mean_class_metrics.csv', index=False)
    CONFUSION_COMPARISON_DF.to_csv(f'{out_dir}/ensemble_confusion_improvement.csv', index=False)
    if MULTI_SEED_FOLD_DF is not None:
        MULTI_SEED_FOLD_DF.to_csv(f'{out_dir}/multi_seed_fold_results.csv', index=False)
        MULTI_SEED_PER_SEED_DF.to_csv(f'{out_dir}/multi_seed_per_seed_summary.csv', index=False)
        MULTI_SEED_SUMMARY_DF.to_csv(f'{out_dir}/multi_seed_summary.csv', index=False)

    for fname in [
        'confusion_matrices_comparison_final.png',
        'fold_comparison_final.png',
        'summary_bars_final.png',
        'multi_seed_summary_final.png',
        'attention_heatmap_final.png',
        'attention_by_day_final.png',
    ]:
        src = f'/content/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{out_dir}/{fname}')
            print(f'Saved: {fname}')
    print(f'All results saved to {out_dir}')
else:
    print('SAVE_TO_DRIVE = False')
    print('Use the Files panel in Colab to download generated CSV and PNG artifacts.')
